# LeetCode 1040 — Moving Stones Until Consecutive II bằng Python

Link đề: https://leetcode.com/problems/moving-stones-until-consecutive-ii/ · Độ khó: **Medium** · Chủ đề: Array, Math, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        ...
```

Ghi chú: đáp án là **cặp `[min, max]` duy nhất**. Trò chơi: mỗi lượt nhấc một **viên đá ở biên** (nhỏ nhất hoặc lớn nhất) và đặt vào ô trống sao cho nó **không còn là biên**; trò chơi kết thúc khi `n` viên nằm ở `n` ô liên tiếp. Oracle độc lập: **duyệt cây trò chơi** (DFS có nhớ) trên mọi trạng thái nhỏ. Lời giải thật dùng công thức: `max` từ khoảng trống, `min` từ **cửa sổ trượt** dài `n` (có một trường hợp đặc biệt).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu luật: chỉ nhấc đá **ở biên**, đặt vào ô trống **nằm giữa** (không còn là biên); kết thúc khi `n` viên liên tiếp; tìm số lượt **ít nhất** và **nhiều nhất**.
2. Nắm công cụ: sắp xếp, hai con trỏ (cửa sổ trượt có kích thước `n` theo giá trị), đếm khoảng trống giữa các viên.
3. Đi từ **duyệt cây trò chơi (DFS có nhớ)** → **công thức `max` + cửa sổ trượt `min`** → **`bisect` thay hai con trỏ** → **liệt kê mọi cửa sổ đích** → **tổng khoảng trống**.
4. Hiểu mấu chốt: `max` = tổng số ô trống **giữa** các viên trừ phần bị mất do lần di chuyển đầu tiên (`max(s[n−1] − s[1], s[n−2] − s[0]) − (n − 2)`); `min` = `n` − số viên lớn nhất nằm trong một **cửa sổ giá trị độ dài `n`**, **trừ** trường hợp `n − 1` viên liên tiếp và viên còn lại ở xa thì cần `2` lượt.
5. Tránh lỗi kinh điển: quên sắp xếp, quên trường hợp đặc biệt của `min` (đáp án `2`), nhầm kích thước cửa sổ, tính `max` chỉ một phía hoặc quên trừ `(n − 2)`, dùng `s[n−1] − s[0] − (n − 1)` (cho phép di chuyển cả hai biên).

## 2. Đề bài & ví dụ

Có một số viên đá trên trục `X`, vị trí cho bởi mảng `stones`. Viên đá **ở biên** là viên có vị trí nhỏ nhất hoặc lớn nhất. Mỗi lượt: nhấc một viên đá ở biên và đặt vào một **ô trống** sao cho nó **không còn là viên biên**.

Ví dụ `stones = [1,2,5]`: không thể di chuyển viên ở vị trí `5` (đặt nó ở `0` hay `3`... hai viên còn lại là `1` và `2`, không có ô nằm giữa), nhưng có thể di chuyển viên `1` vào ô `3` hoặc `4` (nằm giữa `2` và `5`) — chỉ ô nằm **giữa** hai viên còn lại mới hợp lệ.

Trò chơi kết thúc khi không còn lượt đi (các viên ở `n` vị trí liên tiếp).

Trả về mảng `answer` độ dài `2`: `answer[0]` là **số lượt ít nhất**, `answer[1]` là **số lượt nhiều nhất** có thể chơi.

**Ví dụ 1:** `stones = [7,4,9]` → `[1,2]` (`4 → 8` xong trong 1 lượt; hoặc `9 → 5`, `4 → 6` mất 2 lượt).

**Ví dụ 2:** `stones = [6,5,4,3,10]` → `[2,3]`.

**Ràng buộc:** `3 <= stones.length <= 10^4`; `1 <= stones[i] <= 10^9`; các giá trị **khác nhau**.

In [1]:
import bisect
import functools
import itertools
import random
import time
from typing import List

examples = [([7, 4, 9], [1, 2]), ([6, 5, 4, 3, 10], [2, 3]), ([1, 2, 3], [0, 0]), ([1, 2, 5], [2, 2])]
for s, want in examples:
    print(f"stones = {s} -> {want}")

stones = [7, 4, 9] -> [1, 2]
stones = [6, 5, 4, 3, 10] -> [2, 3]
stones = [1, 2, 3] -> [0, 0]
stones = [1, 2, 5] -> [2, 2]


## 3. Kiến thức nền

### 3.1 Luật nước đi

Với tập vị trí `S` đã sắp xếp `s₀ < s₁ < … < s_{n−1}`: chọn viên biên (`s₀` hoặc `s_{n−1}`), gọi `R` là `n − 1` viên còn lại (có min `lo`, max `hi`); đặt viên đó vào ô `p` **chưa có đá** thoả `lo < p < hi` (để không còn là biên). Nếu không có `p` nào (các viên của `R` đã liên tiếp) thì viên đó không di chuyển được.

In [2]:
def legal_moves(state):
    """Mọi trạng thái kế tiếp từ state (tuple đã sắp xếp)."""
    s = list(state)
    n = len(s)
    out = []
    for end in (0, n - 1):
        rest = s[:end] + s[end + 1:]
        lo, hi = rest[0], rest[-1]
        occupied = set(rest)
        for p in range(lo + 1, hi):
            if p not in occupied:
                out.append(tuple(sorted(rest + [p])))
    return out


print(legal_moves((4, 7, 9)))
print("kết thúc (không còn nước đi):", legal_moves((1, 2, 3)), legal_moves((1, 2, 4)))

[(7, 8, 9), (4, 5, 7), (4, 6, 7)]
kết thúc (không còn nước đi): [] [(2, 3, 4)]


### 3.2 Khoảng cách co lại sau mỗi nước

Sau mỗi nước, khoảng `[min, max]` **thu hẹp nghiêm ngặt** (viên biên chuyển vào trong; biên mới là viên kế cạnh). Vậy trò chơi **luôn kết thúc** và đồ thị trạng thái là một **DAG** — ta có thể tính số lượt ít nhất / nhiều nhất bằng DFS có nhớ.

### 3.3 Tính chất: các độ dài có thể có

Ta sẽ kiểm bằng máy (mục 15) rằng mọi độ dài trò chơi trong đoạn `[min, max]` đều đạt được; nhưng đề chỉ hỏi hai đầu mút.

## 4. Ý tưởng nền tảng / trực giác

`stones = [4, 7, 9]` (`n = 3`). Đích: 3 ô liên tiếp. Nước nhanh nhất: `4 → 8` ⇒ `7, 8, 9` (1 lượt). Dài nhất: `9 → 5` (giữa `4` và `7`) rồi `4 → 6` ⇒ `5, 6, 7` (2 lượt).

**Max:** mỗi lượt "tiêu thụ" một ô trống bên trong; tổng ô trống giữa các viên là `(s[n−1] − s[0] + 1) − n`. Lượt đầu tiên phải bỏ một viên biên, khiến ô trống **ngoài vùng còn lại** biến mất: ta chọn bỏ `s₀` (còn `s[1..]`) hoặc `s[n−1]`, rồi mỗi lượt đi chỉ chuyển sang ô trống kề nhất ⇒ `max(s[n−1] − s[1], s[n−2] − s[0]) − (n − 2)`.

**Min:** đích là cửa sổ `n` ô liên tiếp `[t, t + n − 1]`; số viên đá nằm trong cửa sổ càng nhiều thì càng ít lượt (`n − (số viên trong cửa sổ)`), trừ trường hợp đặc biệt.

In [3]:
s = [3, 4, 5, 6, 10]
n = len(s)
print("max =", max(s[-1] - s[1], s[-2] - s[0]) - (n - 2), "(s[-1]-s[1] - (n-2) =", s[-1] - s[1] - (n - 2), "; s[-2]-s[0] - (n-2) =", s[-2] - s[0] - (n - 2), ")")
print("cửa sổ n =", n, "bắt đầu tại mỗi viên:", [(x, sum(1 for y in s if x <= y < x + n)) for x in s])

max = 3 (s[-1]-s[1] - (n-2) = 3 ; s[-2]-s[0] - (n-2) = 0 )
cửa sổ n = 5 bắt đầu tại mỗi viên: [(3, 4), (4, 3), (5, 2), (6, 2), (10, 1)]


## 5. Brute force — duyệt cây trò chơi

**Ý tưởng:** theo đúng luật: `play(state)` trả `(min_moves, max_moves)` từ `state`: nếu không còn nước đi thì `(0, 0)`; ngược lại `1 +` min/max trên các trạng thái kế tiếp. DFS có nhớ (`lru_cache`). Chỉ dùng cho vị trí nhỏ (số trạng thái hữu hạn); là **oracle** độc lập, không dùng công thức.

In [4]:
@functools.lru_cache(maxsize=None)
def play(state):
    nxt = legal_moves(state)
    if not nxt:
        return (0, 0)
    results = [play(t) for t in nxt]
    return (1 + min(r[0] for r in results), 1 + max(r[1] for r in results))


def moves_brute(stones):
    return list(play(tuple(sorted(stones))))


for s, want in examples:
    assert moves_brute(s) == want, (s, moves_brute(s))
print([moves_brute(s) for s, _ in examples])

[[1, 2], [2, 3], [0, 0], [2, 2]]


In [5]:
# Code nộp LeetCode cho Brute force (duyệt cây trò chơi; chỉ cho vị trí nhỏ)
import functools


class SolutionBrute:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        @functools.lru_cache(maxsize=None)
        def play(state):
            s = list(state)
            n = len(s)
            nxt = []
            for end in (0, n - 1):
                rest = s[:end] + s[end + 1:]
                lo, hi = rest[0], rest[-1]
                have = set(rest)
                for p in range(lo + 1, hi):
                    if p not in have:
                        nxt.append(tuple(sorted(rest + [p])))
            if not nxt:
                return (0, 0)
            r = [play(t) for t in nxt]
            return (1 + min(a for a, b in r), 1 + max(b for a, b in r))

        return list(play(tuple(sorted(stones))))

**Độ phức tạp:** số trạng thái `C(khoảng, n)` — mũ; chỉ cho vị trí rất nhỏ.

**Nút thắt:** ta duyệt cả cây trò chơi trong khi đáp án chỉ phụ thuộc vào **khoảng trống** và **cửa sổ** của dãy đã sắp xếp.

## 6. Bước đột phá

Sắp xếp `s`. `max = max(s[n−1] − s[1], s[n−2] − s[0]) − (n − 2)`. `min`: hai con trỏ `i ≤ j` giữ `s[j] − s[i] < n` (cửa sổ giá trị độ dài `n`); `cnt = j − i + 1` là số viên trong cửa sổ; chi phí `n − cnt`, **trừ** trường hợp `cnt = n − 1` và `s[j] − s[i] = n − 2` (đã liên tiếp hoàn toàn) thì chi phí `2`. `min` là nhỏ nhất theo `j`. `O(n log n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** công thức `max` + cửa sổ trượt hai con trỏ (đáp án chính), **Cách 3** `bisect` thay hai con trỏ, **Cách 4** liệt kê cửa sổ đích bằng tập, **Cách 5** tổng khoảng trống.

### Cách 2 — Công thức `max` + cửa sổ trượt

**Giải thích từng đoạn:**

1. `s = sorted(stones)`, `n = len(s)`.
2. `max_moves = max(s[−1] − s[1], s[−2] − s[0]) − (n − 2)`.
3. `min_moves`: `i = 0`; với mỗi `j`: `while s[j] − s[i] ≥ n: i += 1`; `cnt = j − i + 1`; nếu `cnt == n − 1` và `s[j] − s[i] == n − 2`: `cost = 2`, ngược lại `cost = n − cnt`.

In [6]:
def stones_window(stones):
    s = sorted(stones)
    n = len(s)
    max_moves = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    min_moves = n
    i = 0
    for j in range(n):
        while s[j] - s[i] >= n:                          # cửa sổ giá trị [s[i], s[i] + n - 1] phải chứa s[j]
            i += 1
        cnt = j - i + 1
        if cnt == n - 1 and s[j] - s[i] == n - 2:        # n-1 viên liên tiếp, viên còn lại ở xa: cần 2 lượt
            cost = 2
        else:
            cost = n - cnt
        min_moves = min(min_moves, cost)
    return [min_moves, max_moves]


for s, want in examples:
    assert stones_window(s) == want
print([stones_window(s) for s, _ in examples])

[[1, 2], [2, 3], [0, 0], [2, 2]]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionWindow:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        max_moves = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
        min_moves, i = n, 0
        for j in range(n):
            while s[j] - s[i] >= n:
                i += 1
            cnt = j - i + 1
            cost = 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt
            min_moves = min(min_moves, cost)
        return [min_moves, max_moves]

**Độ phức tạp:** `O(n log n)` (sắp xếp), `O(n)` vòng hai con trỏ, `O(n)` bộ nhớ.

### Cách 3 — `bisect` thay hai con trỏ

**Ý tưởng:** với mỗi `i`, số viên trong cửa sổ `[s[i], s[i] + n − 1]` là `bisect_right(s, s[i] + n − 1) − i`; trường hợp đặc biệt xét khi `cnt = n − 1` và viên cuối trong cửa sổ là `s[i] + n − 2`.

In [8]:
def stones_bisect(stones):
    s = sorted(stones)
    n = len(s)
    max_moves = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    min_moves = n
    for i in range(n):
        j = bisect.bisect_right(s, s[i] + n - 1) - 1          # viên cuối cùng nằm trong cửa sổ bắt đầu tại s[i]
        cnt = j - i + 1
        cost = 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt
        min_moves = min(min_moves, cost)
    return [min_moves, max_moves]


for s, want in examples:
    assert stones_bisect(s) == want
print([stones_bisect(s) for s, _ in examples])

[[1, 2], [2, 3], [0, 0], [2, 2]]


In [9]:
# Code nộp LeetCode cho Cách 3
import bisect


class SolutionBisect:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
        mn = n
        for i in range(n):
            j = bisect.bisect_right(s, s[i] + n - 1) - 1
            cnt = j - i + 1
            mn = min(mn, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
        return [mn, mx]

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Liệt kê cửa sổ đích bằng tập

**Ý tưởng:** đích là cửa sổ `n` ô liên tiếp `[t, t + n − 1]`. Chỉ cần thử các `t` mà cửa sổ **chạm** một viên đá ở **đầu** (`t = s[i]`) hoặc ở **cuối** (`t + n − 1 = s[i]`) — các cửa sổ khác kém hơn. Với tập `S`: `cost(t) = n − |S ∩ [t, t + n − 1]|`, rồi áp dụng trường hợp đặc biệt khi `[t, t+n−1]` chứa `n − 1` viên mà viên thiếu nằm ở **ô biên của cửa sổ** (không thể đưa đá vào ô đó trong 1 lượt vì nó bị chặn... ), cụ thể khi `n − 1` viên đã liên tiếp **và** đứng sát một đầu mút xa hơn: trả `2`.

In [10]:
def stones_enumerate(stones):
    S = set(stones)
    s = sorted(stones)
    n = len(s)
    max_moves = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    starts = {x for x in s} | {x - (n - 1) for x in s}                   # cửa sổ bắt đầu tại một viên hoặc kết thúc tại một viên
    best = n
    for t in starts:
        inside = [x for x in s if t <= x <= t + n - 1]
        cnt = len(inside)
        if cnt == n - 1 and inside[-1] - inside[0] == n - 2:             # n-1 viên liên tiếp
            cost = 2
        else:
            cost = n - cnt
        best = min(best, cost)
    return [best, max_moves]


for s, want in examples:
    assert stones_enumerate(s) == want
print([stones_enumerate(s) for s, _ in examples])

[[1, 2], [2, 3], [0, 0], [2, 2]]


In [11]:
# Code nộp LeetCode cho Cách 4 (O(n^2) thử cửa sổ; n tới 10^4 sẽ chậm — chỉ minh hoạ)
class SolutionEnumerate:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
        best = n
        for t in {x for x in s} | {x - (n - 1) for x in s}:
            inside = [x for x in s if t <= x <= t + n - 1]
            cnt = len(inside)
            best = min(best, 2 if cnt == n - 1 and inside[-1] - inside[0] == n - 2 else n - cnt)
        return [best, mx]

**Độ phức tạp:** `O(n²)` thời gian (mỗi cửa sổ duyệt `n` viên), `O(n)` bộ nhớ — chỉ cho `n` nhỏ.

### Cách 5 — Tổng khoảng trống

**Ý tưởng:** `gaps[i] = s[i+1] − s[i] − 1` (số ô trống giữa hai viên liền kề). Khi đó `max = max(sum(gaps[1:]), sum(gaps[:-1]))` (bỏ khoảng trống đầu hoặc cuối). `min` dùng cửa sổ trượt như Cách 2.

In [12]:
def stones_gaps(stones):
    s = sorted(stones)
    n = len(s)
    gaps = [s[k + 1] - s[k] - 1 for k in range(n - 1)]
    max_moves = max(sum(gaps[1:]), sum(gaps[:-1]))                # bỏ khoảng trống đầu hoặc cuối
    i, min_moves = 0, n
    for j in range(n):
        while s[j] - s[i] >= n:
            i += 1
        cnt = j - i + 1
        min_moves = min(min_moves, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
    return [min_moves, max_moves]


for s, want in examples:
    assert stones_gaps(s) == want
print([stones_gaps(s) for s, _ in examples])

[[1, 2], [2, 3], [0, 0], [2, 2]]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionGaps:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        gaps = [s[k + 1] - s[k] - 1 for k in range(n - 1)]
        mx = max(sum(gaps[1:]), sum(gaps[:-1]))
        i, mn = 0, n
        for j in range(n):
            while s[j] - s[i] >= n:
                i += 1
            cnt = j - i + 1
            mn = min(mn, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
        return [mn, mx]

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `stones = [6, 5, 4, 3, 10]` → sắp xếp `s = [3, 4, 5, 6, 10]`, `n = 5`:

- `max = max(10 − 4, 6 − 3) − 3 = max(6, 3) − 3 = 3`.
- Cửa sổ trượt (độ dài giá trị `n = 5`):

| `j` | `s[j]` | `i` | cửa sổ `[s[i], s[i] + 4]` chứa | `cnt` | đặc biệt? | `cost` |
|---|---|---|---|---|---|---|
| 0 | 3 | 0 | `{3}` | 1 | không | 4 |
| 1 | 4 | 0 | `{3,4}` | 2 | không | 3 |
| 2 | 5 | 0 | `{3,4,5}` | 3 | không | 2 |
| 3 | 6 | 0 | `{3,4,5,6}` | 4 = n−1, `6−3 = 3 = n−2` | **có** | 2 |
| 4 | 10 | 3 → `s[i] = 6` | `{6,10}` | 2 | không | 3 |

`min = 2`: kết quả `[2, 3]` ✓. Hàm trace in từng bước:

In [14]:
def trace_window(stones):
    s = sorted(stones)
    n = len(s)
    print("s =", s, ", n =", n)
    print("max =", max(s[-1] - s[1], s[-2] - s[0]) - (n - 2))
    i, best = 0, n
    for j in range(n):
        while s[j] - s[i] >= n:
            i += 1
        cnt = j - i + 1
        special = cnt == n - 1 and s[j] - s[i] == n - 2
        cost = 2 if special else n - cnt
        best = min(best, cost)
        print(f"  j={j}: s[j]={s[j]}, i={i}, cửa sổ [{s[i]},{s[i] + n - 1}] chứa {s[i:j + 1]}, cnt={cnt}, {'ĐẶC BIỆT -> 2' if special else 'cost = n - cnt = ' + str(cost)}")
    print("min =", best)
    return [best, max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)]


assert trace_window([6, 5, 4, 3, 10]) == [2, 3]

s = [3, 4, 5, 6, 10] , n = 5
max = 3
  j=0: s[j]=3, i=0, cửa sổ [3,7] chứa [3], cnt=1, cost = n - cnt = 4
  j=1: s[j]=4, i=0, cửa sổ [3,7] chứa [3, 4], cnt=2, cost = n - cnt = 3
  j=2: s[j]=5, i=0, cửa sổ [3,7] chứa [3, 4, 5], cnt=3, cost = n - cnt = 2
  j=3: s[j]=6, i=0, cửa sổ [3,7] chứa [3, 4, 5, 6], cnt=4, ĐẶC BIỆT -> 2
  j=4: s[j]=10, i=3, cửa sổ [6,10] chứa [6, 10], cnt=2, cost = n - cnt = 3
min = 2


**Trace trường hợp đặc biệt:** `stones = [1, 2, 3, 4, 7]` (`n = 5`): cửa sổ `[1, 5]` chứa `1, 2, 3, 4` (4 viên liên tiếp), viên còn lại `7` ở xa. Công thức thô `n − cnt = 1` sai: viên `7` không thể vào ô `5` vì `5` không nằm **giữa** `1` và `4` (sau khi nhấc `7` các viên còn lại là `1..4`, ô `5` nằm ngoài khoảng `[1, 4]` nên viên đặt vào đó vẫn là biên). Cách đúng cần `2` lượt: `1 → 5` (được vì `5` nằm giữa `2` và `7`), rồi `2 → 6` (nằm giữa `3` và `7`) ⇒ `3,4,5,6,7`. Oracle xác nhận bên dưới.

In [15]:
print("[1,2,3,4,7]: oracle", moves_brute([1, 2, 3, 4, 7]), "| công thức", stones_window([1, 2, 3, 4, 7]))
assert moves_brute([1, 2, 3, 4, 7])[0] == 2 == stones_window([1, 2, 3, 4, 7])[0]

[1,2,3,4,7]: oracle [2, 2] | công thức [2, 2]


## 9. Code nộp LeetCode

Chọn **Cách 2** (công thức + cửa sổ trượt). Giữ nguyên `class Solution`, `numMovesStonesII(self, stones)`:

In [16]:
class Solution:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        max_moves = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)    # bỏ một viên biên, mỗi lượt lấp một ô trống
        min_moves, i = n, 0
        for j in range(n):
            while s[j] - s[i] >= n:                              # cửa sổ giá trị dài n
                i += 1
            cnt = j - i + 1
            if cnt == n - 1 and s[j] - s[i] == n - 2:            # n-1 viên liên tiếp, 1 viên xa: cần 2 lượt
                cost = 2
            else:
                cost = n - cnt
            min_moves = min(min_moves, cost)
        return [min_moves, max_moves]


print([Solution().numMovesStonesII(list(s)) for s, _ in examples])
assert all(Solution().numMovesStonesII(list(s)) == w for s, w in examples)

[[1, 2], [2, 3], [0, 0], [2, 2]]


## 10. Vì sao đúng

**(a) Luật co khoảng.** Mỗi nước chuyển một viên biên vào giữa, nên `[min, max]` thu hẹp nghiêm ngặt. Do đó trò chơi hữu hạn; trạng thái cuối (không còn nước) khi `n − 1` viên còn lại (sau khi bỏ viên biên nào đó) đã liên tiếp **cho cả hai cách bỏ**, tức `n` viên liên tiếp (hoặc `n − 1` viên liên tiếp cộng một viên ngay sát — vẫn là `n` ô liên tiếp).

**(b) `max`.** Gọi `E` là số ô trống trong khoảng `[s₀, s_{n−1}]` = `s_{n−1} − s₀ + 1 − n`. Nước đầu tiên bỏ viên biên (ví dụ `s₀`); khi đó các ô trống **ngoài** đoạn `[s₁, s_{n−1}]` không còn dùng được nữa: mất `s₁ − s₀ − 1` ô. Viên bỏ đi được đặt vào một ô trống trong `(s₁, s_{n−1})`; mỗi nước sau có thể chọn ô trống **gần nhất** bên trong, nên mỗi nước chỉ tiêu thụ **một** ô trống. Số nước tối đa = số ô trống trong `(s₁, s_{n−1})` = `s_{n−1} − s₁ − (n − 2)` (có `n − 1` viên trong khoảng, `n − 2` viên không tính hai đầu... công thức đếm ô trống: `(s_{n−1} − s₁ + 1) − (n − 1) = s_{n−1} − s₁ − (n − 2)`). Đối xứng nếu bỏ `s_{n−1}`; lấy `max` hai cách. Tối ưu vì tổng số ô trống còn dùng được không thể lớn hơn và mỗi nước tiêu thụ ít nhất một ô.

**(c) `min` (cận dưới).** Kết thúc ở `n` ô liên tiếp `[t, t + n − 1]`; mỗi viên **ngoài** cửa sổ phải được chuyển vào ít nhất một lần ⇒ `≥ n − (số viên trong cửa sổ)`. Với cửa sổ tối ưu bất kỳ ta có `min ≥ n − max_cnt`.

**(d) `min` (đạt được).** Với cửa sổ chứa nhiều viên nhất `cnt`, mỗi viên ngoài cửa sổ ở biên có thể được chuyển vào một ô trống của cửa sổ: các viên ngoài luôn là viên **biên** (vì chúng nằm ngoài cửa sổ chứa các viên khác) và ô trống trong cửa sổ nằm giữa các viên còn lại ⇒ hợp lệ. **Ngoại lệ:** nếu `cnt = n − 1` và các viên trong cửa sổ đã liên tiếp (`s[j] − s[i] = n − 2`) thì ô trống duy nhất của cửa sổ nằm ở **mép** (sát viên trong cửa sổ nhưng không nằm giữa hai viên); viên ngoài bị buộc phải đi vào ô đó không hợp lệ (sẽ là biên) ⇒ cần `2` lượt (chuyển một viên biên của dãy liên tiếp ra trước, ...). Khi đó `min` của cửa sổ này là `2`.

**(e) Cách 3, 4, 5** chỉ thay hai con trỏ bằng `bisect`, liệt kê cửa sổ, hay tổng khoảng trống: cùng công thức.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle duyệt cây trò chơi trên **mọi** tập vị trí `n` viên trong `{1..11}` với `n = 3..6` và `{1..14}` với `n = 3..5` (`1 419 + 3 367` trạng thái) (mục 12D), và các cách `O(n log n)` khớp nhau tới `n = 10^4`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Cây trò chơi | mũ (`C(R, n)` trạng thái) | `O(#trạng thái)` | oracle vị trí nhỏ |
| 2. Công thức + hai con trỏ | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. `bisect` | `O(n log n)` | `O(n)` | thay hai con trỏ |
| 4. Liệt kê cửa sổ | `O(n²)` | `O(n)` | minh hoạ ý đích |
| 5. Tổng khoảng trống | `O(n log n)` | `O(n)` | dạng khai báo |

Đo thực tế (`n = 10^4`):

In [17]:
def bench(fn, arg, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1040)
big = rng.sample(range(1, 10 ** 9 + 1), 10000)
dense = rng.sample(range(1, 12001), 10000)
for label, arr in (("ngẫu nhiên thưa", big), ("dày đặc", dense)):
    for name, fn in [("window", stones_window), ("bisect", stones_bisect), ("gaps", stones_gaps)]:
        print(f"{label:16s} {name:7s}: {bench(fn, arr) * 1e3:7.2f} ms -> {fn(arr)}")
print(f"{'enumerate (n=1000)':16s}: {bench(stones_enumerate, big[:1000], repeat=1) * 1e3:7.1f} ms")

ngẫu nhiên thưa  window :    4.58 ms -> [9996, 999663906]


ngẫu nhiên thưa  bisect :    4.08 ms -> [9996, 999663906]


ngẫu nhiên thưa  gaps   :    4.18 ms -> [9996, 999663906]
dày đặc          window :    3.07 ms -> [1651, 2000]


dày đặc          bisect :    6.53 ms -> [1651, 2000]


dày đặc          gaps   :    4.20 ms -> [1651, 2000]


enumerate (n=1000):    78.1 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi tập vị trí nhỏ với oracle cây trò chơi, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "window": stones_window,
    "bisect": stones_bisect,
    "enumerate": stones_enumerate,
    "gaps": stones_gaps,
    "brute": moves_brute,
    "Solution": lambda s: Solution().numMovesStonesII(list(s)),
    "SolutionWindow": lambda s: SolutionWindow().numMovesStonesII(list(s)),
    "SolutionBisect": lambda s: SolutionBisect().numMovesStonesII(list(s)),
    "SolutionEnumerate": lambda s: SolutionEnumerate().numMovesStonesII(list(s)),
    "SolutionGaps": lambda s: SolutionGaps().numMovesStonesII(list(s)),
    "SolutionBrute": lambda s: SolutionBrute().numMovesStonesII(list(s)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(stones, want=None, brute_span=15):
    if want is None and max(stones) - min(stones) <= brute_span:
        want = moves_brute(stones)
    snap = stones[:]
    results = {}
    for name in IMPLS:
        if name in SLOW and max(stones) - min(stones) > brute_span:
            continue
        if name in ("enumerate", "SolutionEnumerate") and len(stones) > 1500:
            continue
        results[name] = IMPLS[name](stones)
    assert stones == snap
    assert len({tuple(v) for v in results.values()}) == 1, (stones[:8], results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (stones[:8], got, want)
    return got


for s, want in examples:
    check(s, want)
print("ví dụ đề: oracle cây trò chơi và", len(IMPLS), "cài đặt đúng, không sửa đầu vào")

ví dụ đề: oracle cây trò chơi và 11 cài đặt đúng, không sửa đầu vào


### B. Edge case

- `n = 3`: `[1,2,3]` → `[0,0]`; `[1,2,4]`; `[1,3,5]`; `[1,2,5]` → `[2,2]`; `[1,2,10]`.
- Đã liên tiếp (`n` viên): `[0, 0]`.
- `n − 1` viên liên tiếp và một viên xa (trường hợp đặc biệt đáp án `2`).
- `n − 1` viên liên tiếp và viên còn lại sát ngay (`[1,2,3,4,6]` → có `[1,2,3,4]` + `6`).
- Hai cụm xa nhau.
- Vị trí thưa (`10^9`): chỉ so khoảng.
- Không sắp xếp sẵn (đầu vào lộn xộn).
- Đối xứng: phản chiếu `p → C − p` không đổi đáp án.

In [19]:
edge = [
    [1, 2, 3], [1, 2, 4], [1, 3, 5], [1, 2, 10], [3, 1, 2], [1, 2, 3, 4, 5], [1, 2, 3, 4, 6], [1, 2, 3, 4, 7], [1, 2, 3, 4, 9], [1, 3, 4, 5, 6], [1, 5, 6, 7, 8], [1, 2, 6, 7, 8, 12],
    [7, 4, 9], [6, 5, 4, 3, 10], [1, 2, 4, 8], [2, 3, 4, 5, 6, 7, 8, 10], [1, 10 ** 9, 5 * 10 ** 8], [1, 2, 10 ** 9], [10 ** 9, 10 ** 9 - 1, 1],
]
for s in edge:
    check(s)
r = Solution().numMovesStonesII([1, 10 ** 9, 5 * 10 ** 8])
print(len(edge), "edge case: mọi cài đặt khớp (oracle cho khoảng ≤ 15) | [1,5e8,1e9] ->", r)
assert r == [2, 499999999]                 # max = max(1e9 - 5e8, 5e8 - 1) - 1 = 499999999
mirror = lambda s, c=20: [c - p for p in s]
for s in ([1, 2, 6, 7, 8, 12], [1, 2, 3, 4, 7], [7, 4, 9]):
    assert Solution().numMovesStonesII(s) == Solution().numMovesStonesII(mirror(s)) == moves_brute(s) == moves_brute(mirror(s))
print("phản chiếu không đổi đáp án: đúng")

19 edge case: mọi cài đặt khớp (oracle cho khoảng ≤ 15) | [1,5e8,1e9] -> [2, 499999999]
phản chiếu không đổi đáp án: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_no_special(stones):                              # min thiếu trường hợp đặc biệt (đáp án 2)
    s = sorted(stones)
    n = len(s)
    mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    i, mn = 0, n
    for j in range(n):
        while s[j] - s[i] >= n:
            i += 1
        mn = min(mn, n - (j - i + 1))
    return [mn, mx]


def bug_max_one_side(stones):                            # max chỉ một phía
    s = sorted(stones)
    n = len(s)
    r = stones_window(stones)
    return [r[0], s[-1] - s[1] - (n - 2)]


def bug_max_no_adjust(stones):                           # quên trừ (n - 2)
    s = sorted(stones)
    r = stones_window(stones)
    return [r[0], max(s[-1] - s[1], s[-2] - s[0])]


def bug_max_both_ends(stones):                           # cho phép bỏ cả hai biên: s[-1] - s[0] - (n - 1)
    s = sorted(stones)
    n = len(s)
    r = stones_window(stones)
    return [r[0], s[-1] - s[0] - (n - 1)]


def bug_window_n_plus_1(stones):                         # cửa sổ giá trị dài n + 1 (s[j] - s[i] > n)
    s = sorted(stones)
    n = len(s)
    mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    i, mn = 0, n
    for j in range(n):
        while s[j] - s[i] > n:
            i += 1
        cnt = j - i + 1
        mn = min(mn, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
    return [mn, mx]


def bug_unsorted(stones):                                # quên sắp xếp
    s = list(stones)
    n = len(s)
    mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    return [stones_window(stones)[0], mx]


def bug_special_too_wide(stones):                        # trường hợp đặc biệt áp dụng cho mọi cnt = n - 1 (không cần liên tiếp)
    s = sorted(stones)
    n = len(s)
    mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
    i, mn = 0, n
    for j in range(n):
        while s[j] - s[i] >= n:
            i += 1
        cnt = j - i + 1
        mn = min(mn, 2 if cnt == n - 1 else n - cnt)
    return [mn, mx]


BUGS = [bug_no_special, bug_max_one_side, bug_max_no_adjust, bug_max_both_ends, bug_window_n_plus_1, bug_unsorted, bug_special_too_wide]
pool = [list(c) for n in range(3, 7) for c in itertools.combinations(range(1, 12), n)]
for bug in BUGS:
    found = None
    for st in pool:
        arr = st[::-1] if bug is bug_unsorted else st
        try:
            got = bug(arr)
        except Exception as e:
            got = type(e).__name__
        if got != moves_brute(st):
            found = (arr, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found[0]}: ra {found[1]}, đúng {moves_brute(found[0])}")

bug_no_special         sai trên [1, 2, 5]: ra [1, 2], đúng [2, 2]
bug_max_one_side       sai trên [1, 3, 4]: ra [1, 0], đúng [1, 1]
bug_max_no_adjust      sai trên [1, 2, 3]: ra [0, 1], đúng [0, 0]
bug_max_both_ends      sai trên [1, 3, 5]: ra [1, 2], đúng [1, 1]
bug_window_n_plus_1    sai trên [1, 2, 4]: ra [0, 1], đúng [1, 1]
bug_unsorted           sai trên [3, 2, 1]: ra [0, -2], đúng [0, 0]
bug_special_too_wide   sai trên [1, 2, 4]: ra [2, 1], đúng [1, 1]


### D. Vét cạn mọi tập vị trí nhỏ

Mọi tập con `n` phần tử của `{1, …, 11}` với `n = 3..6` (`165 + 330 + 462 + 462 = 1 419` tập) và của `{1, …, 14}` với `n = 3..5` (`364 + 1 001 + 2 002 = 3 367` tập). Oracle: DFS có nhớ trên cây trò chơi (định nghĩa luật). Mọi cài đặt phải khớp (riêng bản nộp `SolutionBrute` — tự tạo bộ nhớ đệm mỗi lần gọi — thử trên mẫu).

In [21]:
t0 = time.perf_counter()
cnt = 0
for N, sizes in ((11, range(3, 7)), (14, range(3, 6))):
    for n in sizes:
        for c in itertools.combinations(range(1, N + 1), n):
            st = list(c)
            want = moves_brute(st)
            for name in IMPLS:
                if name == "SolutionBrute":                      # bản nộp tự tạo bộ nhớ đệm mỗi lần gọi: chỉ thử trên mẫu (xem bên dưới)
                    continue
                got = IMPLS[name](st)
                assert got == want, (name, st, got, want)
            cnt += 1
print(cnt, "tập vị trí (n 3..6 trong {1..11}; n 3..5 trong {1..14}):", len(IMPLS), "cài đặt khớp oracle cây trò chơi;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 1419 + 3367
rng = random.Random(104002)
all_small = [list(c) for n in range(3, 6) for c in itertools.combinations(range(1, 10), n)]
for st in rng.sample(all_small, 150):
    assert SolutionBrute().numMovesStonesII(st) == moves_brute(st)
print("SolutionBrute (bản nộp, tự tạo bộ nhớ đệm mỗi lần gọi) khớp trên 150 tập mẫu")

4786 tập vị trí (n 3..6 trong {1..11}; n 3..5 trong {1..14}): 11 cài đặt khớp oracle cây trò chơi; 0.1 s
SolutionBrute (bản nộp, tự tạo bộ nhớ đệm mỗi lần gọi) khớp trên 150 tập mẫu


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách nhanh khớp nhau (và `enumerate` với `n ≤ 1500`); (2) `0 ≤ min ≤ max`; (3) `min ≤ 2` hoặc `min = n − cnt_max`... kiểm `min ≤ n − 1`; (4) **dịch** tất cả vị trí cùng hằng không đổi đáp án; (5) **phản chiếu** không đổi; (6) thứ tự đầu vào không ảnh hưởng (xáo trộn); (7) `min = 0 ⇔ max = 0 ⇔` đã liên tiếp.

In [22]:
rng = random.Random(20240119)
t0 = time.perf_counter()
fastn = ["window", "bisect", "gaps", "Solution", "SolutionWindow", "SolutionBisect", "SolutionGaps"]
for it in range(1500):
    n = rng.choice([3, 4, 5, 8, 20, 100, 500, 2000, 10000])
    span = rng.choice([n + 1, n + 5, 3 * n, 10 ** 9])
    span = max(span, n)
    st = rng.sample(range(1, span + 1), n) if span <= 10 ** 6 else [rng.randint(1, span) for _ in range(n)]
    st = list(dict.fromkeys(st))
    if len(st) < 3:
        continue
    n = len(st)
    res = {name: IMPLS[name](st) for name in fastn}
    assert len({tuple(v) for v in res.values()}) == 1, (n, res)                          # (1)
    mn, mx = res["Solution"]
    if n <= 1500:
        assert IMPLS["enumerate"](st) == [mn, mx]                                        # (1)
    assert 0 <= mn <= mx and mn <= n - 1                                                 # (2) (3)
    shift = rng.randint(0, 5)
    assert Solution().numMovesStonesII([p + shift for p in st]) == [mn, mx]              # (4)
    csum = max(st) + min(st)
    assert Solution().numMovesStonesII([csum - p for p in st]) == [mn, mx]               # (5)
    sh = st[:]
    rng.shuffle(sh)
    assert Solution().numMovesStonesII(sh) == [mn, mx]                                   # (6)
    consecutive = max(st) - min(st) == n - 1
    assert (mn == 0) == consecutive and (mx == 0) == consecutive                         # (7)
print("1500 tập ngẫu nhiên (n tới 10^4, vị trí tới 10^9): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

1500 tập ngẫu nhiên (n tới 10^4, vị trí tới 10^9): 7 tính chất đúng; 13.8 s


### F. Giới hạn

`n = 10^4`: toàn liên tiếp (`[0, 0]`); `n − 1` viên liên tiếp và một viên xa (`min = 2`); hai cụm; vị trí thưa tới `10^9`; vị trí chèn dày (`10^4` giá trị trong `12 000` ô); các cách `O(n log n)` khớp nhau.

In [23]:
t0 = time.perf_counter()
n = 10000
rng = random.Random(104001)
consec = list(range(5, 5 + n))
almost = list(range(1, n)) + [10 ** 9]
two_clusters = list(range(1, n // 2 + 1)) + list(range(10 ** 9 - n // 2, 10 ** 9))
sparse = rng.sample(range(1, 10 ** 9 + 1), n)
dense = rng.sample(range(1, 12001), n)
for name, arr, want in [("liên tiếp", consec, [0, 0]), ("n-1 liên tiếp + 1 xa", almost, None), ("hai cụm", two_clusters, None), ("thưa", sparse, None), ("dày", dense, None)]:
    res = {k: IMPLS[k](arr) for k in fastn}
    assert len({tuple(v) for v in res.values()}) == 1, (name, res)
    if want is not None:
        assert res["Solution"] == want
    print(f"  {name:22s}: {res['Solution']}")
assert Solution().numMovesStonesII(almost)[0] == 2
print("n = 10^4: các cách khớp nhau; 'n-1 liên tiếp + 1 xa' cho min = 2;", round(time.perf_counter() - t0, 2), "s")

  liên tiếp             : [0, 0]
  n-1 liên tiếp + 1 xa  : [2, 999990000]
  hai cụm               : [5000, 999989999]
  thưa                  : [9996, 999835473]
  dày                   : [1630, 2000]
n = 10^4: các cách khớp nhau; 'n-1 liên tiếp + 1 xa' cho min = 2; 0.14 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên trường hợp đặc biệt của `min`

In [24]:
s = [1, 2, 3, 4, 7]
print("bug:", bug_no_special(s), "| đúng:", Solution().numMovesStonesII(s), "| oracle:", moves_brute(s))
assert bug_no_special(s)[0] == 1 and Solution().numMovesStonesII(s)[0] == 2

bug: [1, 2] | đúng: [2, 2] | oracle: [2, 2]


**Vì sao sai:** với `n − 1` viên đã liên tiếp và viên còn lại ở xa, ô trống duy nhất trong cửa sổ không nằm **giữa** hai viên khác, nên không thể lấp bằng một lượt hợp lệ ⇒ `2`.

### Lỗi 2: `max` chỉ một phía hoặc thiếu `(n − 2)`

In [25]:
s = [1, 2, 6, 7, 8, 12]
print("một phía:", bug_max_one_side(s), "| thiếu -(n-2):", bug_max_no_adjust(s), "| đúng:", Solution().numMovesStonesII(s))
bad = next(st for st in pool if bug_max_one_side(st) != moves_brute(st))
print("phản ví dụ một phía:", bad, bug_max_one_side(bad), "đúng", moves_brute(bad))
assert bug_max_no_adjust(s) != Solution().numMovesStonesII(s)

một phía: [3, 6] | thiếu -(n-2): [3, 10] | đúng: [3, 6]
phản ví dụ một phía: [1, 3, 4] [1, 0] đúng [1, 1]


**Vì sao sai:** hai lựa chọn bỏ biên (`s₀` hay `s_{n−1}`) cho giá trị khác nhau; và đếm **ô trống** cần trừ `(n − 2)` viên còn nằm trong khoảng.

### Lỗi 3: cho phép bỏ cả hai biên (`s[-1] − s[0] − (n − 1)`)

In [26]:
bad = next(st for st in pool if bug_max_both_ends(st) != moves_brute(st))
print("phản ví dụ:", bad, "| bug:", bug_max_both_ends(bad), "| đúng:", moves_brute(bad))
assert bug_max_both_ends(bad) != moves_brute(bad)

phản ví dụ: [1, 3, 5] | bug: [1, 2] | đúng: [1, 1]


**Vì sao sai:** sau nước đầu tiên khoảng trống ngoài vùng còn lại biến mất; số ô trống toàn bộ không phải số lượt tối đa.

### Lỗi 4: sai kích thước cửa sổ

In [27]:
bad = next(st for st in pool if bug_window_n_plus_1(st) != moves_brute(st))
print("phản ví dụ:", bad, "| bug:", bug_window_n_plus_1(bad), "| đúng:", moves_brute(bad))
assert bug_window_n_plus_1(bad) != moves_brute(bad)

phản ví dụ: [1, 2, 4] | bug: [0, 1] | đúng: [1, 1]


**Vì sao sai:** đích có đúng `n` ô liên tiếp: cửa sổ giá trị là `[s[i], s[i] + n − 1]` (độ dài `n`), tức điều kiện `s[j] − s[i] ≤ n − 1`.

### Lỗi 5: quên sắp xếp

In [28]:
st = [6, 5, 4, 3, 10]
print("bug:", bug_unsorted(st), "| đúng:", Solution().numMovesStonesII(st))
bad = next(c for c in pool if bug_unsorted(c[::-1]) != moves_brute(c))
print("phản ví dụ (đảo thứ tự):", bad[::-1], "-> bug", bug_unsorted(bad[::-1]), "| đúng:", moves_brute(bad))
assert bug_unsorted(bad[::-1]) != moves_brute(bad)

bug: [2, 2] | đúng: [2, 3]
phản ví dụ (đảo thứ tự): [3, 2, 1] -> bug [0, -2] | đúng: [0, 0]


**Vì sao sai:** công thức dựa trên `s[0]`, `s[1]`, `s[n−2]`, `s[n−1]` là các phần tử **nhỏ/lớn nhất** nên phải sắp xếp.

### Lỗi 6 (bonus): trường hợp đặc biệt áp dụng quá rộng

In [29]:
bad = next(st for st in pool if bug_special_too_wide(st) != moves_brute(st))
print("phản ví dụ:", bad, "| bug:", bug_special_too_wide(bad), "| đúng:", moves_brute(bad))
assert bug_special_too_wide(bad) != moves_brute(bad)

phản ví dụ: [1, 2, 4] | bug: [2, 1] | đúng: [1, 1]


**Vì sao sai:** ngoại lệ chỉ xảy ra khi `n − 1` viên **đã liên tiếp** (`s[j] − s[i] = n − 2`); nếu có khe trong cửa sổ thì có ô trống nằm giữa và một lượt là đủ.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "trò chơi di chuyển phần tử để gom thành đoạn liên tiếp, số lượt ít/nhiều nhất" → `max` từ khoảng trống, `min` từ **cửa sổ giá trị** dài `n` (hai con trỏ) + ngoại lệ.

**Mẹo:**

1. Sắp xếp; làm việc với `s[0], s[1], s[n−2], s[n−1]`.
2. Cửa sổ theo **giá trị** (không theo chỉ số): `s[j] − s[i] ≤ n − 1`.
3. Ngoại lệ của `min`: `n − 1` viên liên tiếp và 1 viên xa ⇒ `2`.
4. `max` = ô trống trong vùng còn lại sau khi bỏ một viên biên.
5. Oracle: DFS có nhớ trên cây trò chơi (hữu hạn vì khoảng co lại).
6. Kiểm tra bất biến: dịch, phản chiếu.

**Họ bài cùng pattern:** Moving Stones Until Consecutive (1033), Minimum Number of Moves to Make Palindrome (2193), Minimum Moves to Equal Array Elements (453), Minimum Swaps to Group All 1's Together (1151), Minimum Adjacent Swaps for K Consecutive Ones (1703), Sliding Window Maximum (239).

## 15. Biến thể

1. **Chỉ phụ thuộc chuỗi khoảng trống:** đáp án chỉ phụ thuộc vào dãy `gaps = [s[k+1] − s[k] − 1]` (và đảo ngược của nó).
2. **Mọi độ dài trò chơi trong `[min, max]` đều đạt được?** — kiểm bằng cây trò chơi.
3. **Kết thúc ở cửa sổ nào:** tập các cửa sổ đích có thể đạt tới.

Cài đặt và đối chiếu oracle:

In [30]:
@functools.lru_cache(maxsize=None)
def lengths(state):
    nxt = legal_moves(state)
    if not nxt:
        return frozenset({0})
    return frozenset(1 + x for t in nxt for x in lengths(t))


@functools.lru_cache(maxsize=None)
def terminal_windows(state):
    nxt = legal_moves(state)
    if not nxt:
        return frozenset({state[0]})
    return frozenset(w for t in nxt for w in terminal_windows(t))


def gap_signature(stones):
    s = sorted(stones)
    gaps = tuple(s[k + 1] - s[k] - 1 for k in range(len(s) - 1))
    return min(gaps, gaps[::-1])


by_sig = {}
cnt = 0
for n in range(3, 7):
    for st in itertools.combinations(range(1, 13), n):
        L = lengths(st)
        assert L == frozenset(range(min(L), max(L) + 1)), st                       # (2) mọi số lượt trong [min, max] đều đạt được
        assert [min(L), max(L)] == moves_brute(list(st))
        sig = gap_signature(st)
        by_sig.setdefault(sig, set()).add(tuple(moves_brute(list(st))))            # (1) cùng chữ ký khoảng trống => cùng đáp án
        W = terminal_windows(st)
        assert all(w + n - 1 <= st[-1] and w >= st[0] for w in W)                  # (3) cửa sổ đích nằm trong [min, max] ban đầu
        cnt += 1
assert all(len(v) == 1 for v in by_sig.values())
print(cnt, "trạng thái: mọi số lượt trong [min, max] đều đạt được;", len(by_sig), "chữ ký khoảng trống, mỗi chữ ký cho đúng một đáp án")

2431 trạng thái: mọi số lượt trong [min, max] đều đạt được; 541 chữ ký khoảng trống, mỗi chữ ký cho đúng một đáp án


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cây trò chơi | định nghĩa | mũ | `O(#trạng thái)` | oracle vị trí nhỏ |
| 2. Công thức + hai con trỏ | `max` khoảng trống, `min` cửa sổ | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. `bisect` | cửa sổ qua nhị phân | `O(n log n)` | `O(n)` | thay hai con trỏ |
| 4. Liệt kê cửa sổ | thử mọi cửa sổ đích | `O(n²)` | `O(n)` | minh hoạ ý đích |
| 5. Tổng khoảng trống | `max` qua `gaps` | `O(n log n)` | `O(n)` | dạng khai báo |

Mọi cách khớp oracle cây trò chơi trên `4 786` tập vị trí nhỏ và khớp nhau tới `n = 10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().numMovesStonesII([7, 4, 9]), Solution().numMovesStonesII([6, 5, 4, 3, 10]), Solution().numMovesStonesII([1, 2, 3]),
      Solution().numMovesStonesII([1, 2, 5]), Solution().numMovesStonesII([1, 2, 3, 4, 7]))

[1, 2] [2, 3] [0, 0] [2, 2] [2, 2]


### Bài 2 (tính tay)

`stones = [1, 2, 4, 8]`. (a) `max`. (b) Các cửa sổ giá trị dài `4` và số viên trong mỗi cửa sổ; `min`. (c) Kiểm bằng oracle ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cửa sổ trượt tính `min`:

In [32]:
def min_todo(stones):
    s = sorted(stones)
    n = len(s)
    i, best = 0, n
    for j in range(n):
        pass            # TODO 1: while s[j] - s[i] >= n: i += 1
        pass            # TODO 2: cnt = j - i + 1; cost = 2 nếu (cnt == n - 1 và s[j] - s[i] == n - 2) ngược lại n - cnt
        pass            # TODO 3: best = min(best, cost)
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(min_todo(s) == w[0] for s, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`bisect`** (tính `min` qua `bisect_right(s, s[i] + n − 1)`):

In [33]:
class MySolution:
    def numMovesStonesII(self, stones):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().numMovesStonesII(list(s)) == w for s, w in examples)

### Bài 5 (mini challenge): chữ ký khoảng trống

Viết `gap_key(stones)` trả về `min(gaps, gaps[::-1])` với `gaps = (s[k+1] − s[k] − 1)` (dãy khoảng trống giữa các viên đã sắp xếp, hoặc đảo ngược của nó, lấy nhỏ hơn). Kiểm tra rằng hai tập có cùng `gap_key` luôn có cùng đáp án (`moves_brute`).

In [34]:
def gap_key(stones):
    pass            # TODO


# Khi làm xong:
# assert gap_key([1, 2, 6]) == gap_key([10, 14, 15]) and moves_brute([1, 2, 6]) == moves_brute([10, 14, 15])

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1, 2 cho `[1,2]`, `[2,3]`.
- Mức 2: `[1,2,3]` đã liên tiếp ⇒ `[0,0]`; `[1,2,5]`: viên `5` không di chuyển được ngay (không có ô giữa `1` và `2`); phải dời `1 → 3` hoặc `4` trước.
- Mức 3: `[1, 2] [2, 3] [0, 0] [2, 2] [2, 2]`.

**Bài 2**

- Mức 1: `s = [1, 2, 4, 8]`, `n = 4`: `max = max(8 − 2, 4 − 1) − 2 = 4`.
- Mức 2: cửa sổ `[1,4]` chứa `{1,2,4}` (3 viên, không liên tiếp `4 − 1 = 3 ≠ n − 2 = 2`) ⇒ cost `1`.
- Mức 3: `min = 1`, `max = 4` ⇒ `[1, 4]`.

**Bài 3**

- Mức 1: `while s[j] - s[i] >= n: i += 1`.
- Mức 2: `cnt = j - i + 1`; đặc biệt khi `cnt == n - 1 and s[j] - s[i] == n - 2`.
- Mức 3: `best = min(best, cost)`.

**Bài 4**

- Mức 1: `j = bisect.bisect_right(s, s[i] + n - 1) - 1`.
- Mức 2: `cnt = j - i + 1`.
- Mức 3: cùng công thức với trường hợp đặc biệt.

**Bài 5**

- Mức 1: sắp xếp `s`; `gaps[k] = s[k+1] - s[k] - 1`.
- Mức 2: phản chiếu đảo ngược thứ tự `gaps`; dịch chuyển không đổi `gaps`.
- Mức 3: `return min(tuple(gaps), tuple(gaps[::-1]))`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().numMovesStonesII([7, 4, 9]), Solution().numMovesStonesII([6, 5, 4, 3, 10]), Solution().numMovesStonesII([1, 2, 3]),
        Solution().numMovesStonesII([1, 2, 5]), Solution().numMovesStonesII([1, 2, 3, 4, 7]))
print(vals)
assert vals == ([1, 2], [2, 3], [0, 0], [2, 2], [2, 2])
for s, v in zip(([7, 4, 9], [6, 5, 4, 3, 10], [1, 2, 3], [1, 2, 5], [1, 2, 3, 4, 7]), vals):
    assert moves_brute(s) == v
print("Bài 1 OK")

([1, 2], [2, 3], [0, 0], [2, 2], [2, 2])
Bài 1 OK


### Bài 2

In [36]:
s = [1, 2, 4, 8]
n = len(s)
mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
windows = [(x, [y for y in s if x <= y <= x + n - 1]) for x in s]
print("max =", mx, "| cửa sổ:", windows)
assert mx == 4 and Solution().numMovesStonesII(s) == moves_brute(s) == [1, 4]
print("Bài 2 OK: [1, 4]")

max = 4 | cửa sổ: [(1, [1, 2, 4]), (2, [2, 4]), (4, [4]), (8, [8])]
Bài 2 OK: [1, 4]


### Bài 3

In [37]:
def min_answer(stones):
    s = sorted(stones)
    n = len(s)
    i, best = 0, n
    for j in range(n):
        while s[j] - s[i] >= n:
            i += 1
        cnt = j - i + 1
        cost = 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt
        best = min(best, cost)
    return best


assert all(min_answer(s) == w[0] for s, w in examples)
for st in itertools.combinations(range(1, 12), 5):
    assert min_answer(list(st)) == moves_brute(list(st))[0]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def numMovesStonesII(self, stones):
        s = sorted(stones)
        n = len(s)
        mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
        mn = n
        for i in range(n):
            j = bisect.bisect_right(s, s[i] + n - 1) - 1
            cnt = j - i + 1
            mn = min(mn, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
        return [mn, mx]


assert all(MySolutionAnswer().numMovesStonesII(list(s)) == w for s, w in examples)
for st in itertools.combinations(range(1, 12), 4):
    assert MySolutionAnswer().numMovesStonesII(list(st)) == moves_brute(list(st))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def gap_key(stones):
    t = sorted(stones)
    gaps = tuple(t[k + 1] - t[k] - 1 for k in range(len(t) - 1))
    return min(gaps, gaps[::-1])


assert gap_key([1, 2, 6]) == gap_key([10, 14, 15]) and moves_brute([1, 2, 6]) == moves_brute([10, 14, 15])
table = {}
for n in range(3, 6):
    for st in itertools.combinations(range(1, 12), n):
        table.setdefault(gap_key(st), set()).add(tuple(moves_brute(list(st))))
assert all(len(v) == 1 for v in table.values())
print("Bài 5 OK:", len(table), "chữ ký, mỗi chữ ký một đáp án")

Bài 5 OK: 205 chữ ký, mỗi chữ ký một đáp án


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (gom đá thành đoạn liên tiếp: `max` từ khoảng trống, `min` từ cửa sổ giá trị dài `n`):**

```python
class Solution:
    def numMovesStonesII(self, stones: List[int]) -> List[int]:
        s = sorted(stones)
        n = len(s)
        mx = max(s[-1] - s[1], s[-2] - s[0]) - (n - 2)
        mn, i = n, 0
        for j in range(n):
            while s[j] - s[i] >= n:
                i += 1
            cnt = j - i + 1
            mn = min(mn, 2 if cnt == n - 1 and s[j] - s[i] == n - 2 else n - cnt)
        return [mn, mx]
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** `stones`?
- [ ] `max = max(s[-1] − s[1], s[-2] − s[0]) − (n − 2)` (hai phía, có trừ `n − 2`)?
- [ ] Cửa sổ giá trị dài `n`: `while s[j] − s[i] >= n`?
- [ ] Trường hợp đặc biệt: `cnt == n − 1` và `s[j] − s[i] == n − 2` ⇒ chi phí `2`?
- [ ] Trả `[min, max]` đúng thứ tự?
- [ ] Đã thử `[7,4,9]`, `[6,5,4,3,10]`, `[1,2,3,4,7]` (min 2), đã liên tiếp (`[0,0]`)?
- [ ] Đúng tên `class Solution`, `numMovesStonesII(self, stones)`, không in debug.